## 5 — SRP (county-level estimates)
Stacked Regression Poststratification at the county level.

Five base learners:
1. **HLM** — Howe (2015) 4-level logistic MAP: county effects ~ N(state + γ·county-covariates, σ²)
2. LASSO — elastic net with CV-tuned λ (one-hot demographics, label-encoded county_fips)
3. KNN — k-nearest neighbours, label-encoded county_fips (~3,143 levels)
4. Random Forest — label encoding
5. XGBoost — label encoding

Stack weights learned by 5-fold cross-validated NNLS (non-negative, sum-to-one).

Output: `outputs/estimates/srp_county_estimates.csv`

In [1]:
get_ipython().run_line_magic("pip", "install xgboost -q")

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.optimize import minimize, nnls
from scipy.special import expit
from sklearn.linear_model import LassoCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
import xgboost as xgb

DATA_DIR   = Path("../test_data/processed/")
OUTPUT_DIR = Path("../outputs/")
OUTCOME    = "happening_bin"
MODEL_NAME = "srp"
STATE_CSV_NAME = "srp_state_estimates.csv"
SEED    = 42
N_FOLDS = 5

STATE_NAMES = {
    "01":"Alabama","02":"Alaska","04":"Arizona","05":"Arkansas","06":"California",
    "08":"Colorado","09":"Connecticut","10":"Delaware","11":"District of Columbia",
    "12":"Florida","13":"Georgia","15":"Hawaii","16":"Idaho","17":"Illinois",
    "18":"Indiana","19":"Iowa","20":"Kansas","21":"Kentucky","22":"Louisiana",
    "23":"Maine","24":"Maryland","25":"Massachusetts","26":"Michigan",
    "27":"Minnesota","28":"Mississippi","29":"Missouri","30":"Montana",
    "31":"Nebraska","32":"Nevada","33":"New Hampshire","34":"New Jersey",
    "35":"New Mexico","36":"New York","37":"North Carolina","38":"North Dakota",
    "39":"Ohio","40":"Oklahoma","41":"Oregon","42":"Pennsylvania",
    "44":"Rhode Island","45":"South Carolina","46":"South Dakota",
    "47":"Tennessee","48":"Texas","49":"Utah","50":"Vermont",
    "51":"Virginia","53":"Washington","54":"West Virginia","55":"Wisconsin",
    "56":"Wyoming",
}

In [3]:
# ── Build county-level covariate table ─────────────────────────────────────
# poststrat_county already carries co2_per_capita and dem_share_two_party at the
# county level; merge in pct_drive_alone, pct_samesex_hh from the ACS extras.
ps_county = pd.read_csv(DATA_DIR / "poststrat_county.csv",
                         dtype={"county_fips": str, "state_fips": str})
acs_county = pd.read_csv(DATA_DIR / "acs_county_extra_covariates.csv",
                          dtype={"county_fips": str, "state_fips": str})

ps_county = ps_county.merge(
    acs_county[["county_fips", "pct_drive_alone", "pct_samesex_hh"]],
    on="county_fips", how="left",
)

county_cov = (
    ps_county[["county_fips", "state_fips",
               "co2_per_capita", "dem_share_two_party",
               "pct_drive_alone", "pct_samesex_hh"]]
    .drop_duplicates(subset=["county_fips"])
    .reset_index(drop=True)
)
# Impute any missing covariate values with the national mean
for c in ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]:
    county_cov[c] = county_cov[c].fillna(county_cov[c].mean())

print(f"poststrat_county rows: {len(ps_county):,}  | counties: {ps_county['county_fips'].nunique():,}")
print(f"county covariate table: {county_cov.shape}")
print(county_cov.head(4).to_string(index=False))

poststrat_county rows: 99,940  | counties: 3,143
county covariate table: (3143, 6)
county_fips state_fips  co2_per_capita  dem_share_two_party  pct_drive_alone  pct_samesex_hh
      01001         01       81.134604             0.266411         0.843044        0.005769
      01003         01       10.888320             0.206524         0.794437        0.005769
      01005         01       12.489276             0.425850         0.832330        0.005769
      01007         01       10.493996             0.176151         0.848439        0.005769


In [4]:
# ── Load survey ───────────────────────────────────────────────────────────
survey = pd.read_csv(DATA_DIR / "climate_survey_responses_recoded.csv",
                     dtype={"state_fips": str, "county_fips": str})
survey = survey.dropna(subset=[OUTCOME]).copy()
survey[OUTCOME] = survey[OUTCOME].astype(float)
survey["educ_category"] = survey["educ_category"].astype(str)

# Merge per-respondent county covariates (used as numeric features by LASSO/KNN/RF/XGB)
survey = survey.merge(county_cov.drop(columns=["state_fips"]),
                       on="county_fips", how="left")
for c in ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]:
    survey[c] = survey[c].fillna(county_cov[c].mean())

print(f"Survey: {len(survey):,}  ({survey[OUTCOME].mean()*100:.1f}% Yes)")

Survey: 1,011  (57.9% Yes)


In [5]:
# ── Build hierarchical indices ────────────────────────────────────────────
# county_cats: ALL counties from poststrat_county (~3,143)
# state_cats : ALL states  from poststrat_county (~51)
# div_cats   : 9 Census divisions (sourced from poststrat_county[DIVISION])
county_cats = sorted(ps_county["county_fips"].unique())
state_cats  = sorted(ps_county["state_fips"].unique())

# Each county_fips → its parent state_fips (ordered by county_cats)
cc = (county_cov.set_index("county_fips").loc[county_cats].reset_index())

county_state_idx = pd.Categorical(
    cc["state_fips"].astype(str),
    categories=state_cats,
).codes

# Each state_fips → its Census division (from poststrat_county.DIVISION)
state_div = (ps_county.groupby("state_fips")["DIVISION"].first()
                       .reset_index().rename(columns={"DIVISION": "division"}))
div_cats = sorted(state_div["division"].unique())
n_div = len(div_cats)

state_div_idx = pd.Categorical(
    state_div.set_index("state_fips").loc[state_cats]["division"].astype(str).values,
    categories=[str(d) for d in div_cats],
).codes

# Standardised county-level covariates (z-score, ordered to match county_cats)
def _std(x):
    return (x - x.mean()) / x.std()

co2_std     = _std(cc["co2_per_capita"].values)
pres_std    = _std(cc["dem_share_two_party"].values)
drive_std   = _std(cc["pct_drive_alone"].values)
samesex_std = _std(cc["pct_samesex_hh"].values)

n_county = len(county_cats)
n_s      = len(state_cats)

print(f"Counties: {n_county:,}  |  States: {n_s}  |  Divisions: {n_div}")
print(f"co2_std  range: [{co2_std.min():.2f}, {co2_std.max():.2f}]")
print(f"drive_std range: [{drive_std.min():.2f}, {drive_std.max():.2f}]")

Counties: 3,143  |  States: 51  |  Divisions: 9
co2_std  range: [-0.08, 53.72]
drive_std range: [-8.98, 2.53]


In [6]:
# ── Encode survey for the HLM block ───────────────────────────────────────
gender_cats = sorted(survey["gender"].unique())
race_cats   = sorted(survey["race4"].unique())
educ_cats   = sorted(survey["educ_category"].unique())

g_idx_full = pd.Categorical(survey["gender"],        categories=gender_cats).codes
r_idx_full = pd.Categorical(survey["race4"],         categories=race_cats).codes
e_idx_full = pd.Categorical(survey["educ_category"], categories=educ_cats).codes
c_idx_full = pd.Categorical(survey["county_fips"],   categories=county_cats).codes

# Drop survey rows whose county isn't in poststrat_county (c_idx == -1)
mask = c_idx_full >= 0
if (~mask).sum():
    print(f"Dropping {(~mask).sum()} survey rows: county_fips not in poststrat_county")
    survey = survey.loc[mask].reset_index(drop=True)
    g_idx_full = g_idx_full[mask]
    r_idx_full = r_idx_full[mask]
    e_idx_full = e_idx_full[mask]
    c_idx_full = c_idx_full[mask]

y = survey[OUTCOME].values.astype(float)
n_g, n_r, n_e = len(gender_cats), len(race_cats), len(educ_cats)
N_GROUPS = [n_g, n_r, n_e, n_county, n_s]

# Survey feature matrices (label-encoded indices for tree learners; one-hot for LASSO)
X_label_demog = np.column_stack([g_idx_full, r_idx_full, e_idx_full, c_idx_full])

# Numeric covariates (already merged onto survey)
COV_COLS = ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]
X_cov = survey[COV_COLS].values

# Combined label feature matrix used by HLM/RF/XGB
X_label = np.column_stack([X_label_demog, X_cov]).astype(float)

# One-hot for LASSO: demographics one-hot + county_fips label-encoded + numeric covariates
ohe_demog = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_ohe_demog = ohe_demog.fit_transform(survey[["gender", "race4", "educ_category"]])
X_ohe = np.hstack([X_ohe_demog,
                    c_idx_full.reshape(-1, 1).astype(float),
                    X_cov])

print(f"Label-encoded shape: {X_label.shape}")
print(f"One-hot+county shape: {X_ohe.shape}")

Dropping 2 survey rows: county_fips not in poststrat_county
Label-encoded shape: (1009, 8)
One-hot+county shape: (1009, 15)


In [7]:
# ── Build poststrat feature matrices in matching column order ─────────────
ps = ps_county[["county_fips", "state_fips", "gender", "race4", "educ_category",
                  "N_rounded", "co2_per_capita", "dem_share_two_party",
                  "pct_drive_alone", "pct_samesex_hh"]].copy()
ps["educ_category"] = ps["educ_category"].astype(str)
for c in COV_COLS:
    ps[c] = ps[c].fillna(county_cov[c].mean())

ps_g = pd.Categorical(ps["gender"],        categories=gender_cats).codes
ps_r = pd.Categorical(ps["race4"],         categories=race_cats).codes
ps_e = pd.Categorical(ps["educ_category"], categories=educ_cats).codes
ps_c = pd.Categorical(ps["county_fips"],   categories=county_cats).codes

ps_label = np.column_stack([ps_g, ps_r, ps_e, ps_c, ps[COV_COLS].values]).astype(float)

ps_ohe_demog = ohe_demog.transform(ps[["gender", "race4", "educ_category"]])
ps_ohe = np.hstack([ps_ohe_demog,
                     ps_c.reshape(-1, 1).astype(float),
                     ps[COV_COLS].values])

print(f"Poststrat label shape: {ps_label.shape}")
print(f"Poststrat one-hot shape: {ps_ohe.shape}")

Poststrat label shape: (99940, 8)
Poststrat one-hot shape: (99940, 15)


### Base learner helpers
Each `fit_*` function returns a callable `predict(X)` so the stacking loop is uniform.

In [8]:
# ── HLM: Howe (2015) 4-level logistic MAP (county architecture) ──────────
# Uses globals: county_state_idx, state_div_idx, co2_std, pres_std, drive_std,
# samesex_std, n_county, n_s, n_div
def fit_hlm(y_tr, gi, ri, ei, ci, n_groups):
    ng, nr, ne, ncnty, nstate = n_groups

    def _unpack(p):
        g0    = p[0]
        sig   = np.exp(p[1:7])
        gc, gp, gd, gs = p[7], p[8], p[9], p[10]
        off = 11
        ug    = p[off:off+ng];      off += ng
        ur    = p[off:off+nr];      off += nr
        ue    = p[off:off+ne];      off += ne
        ureg  = p[off:off+n_div];   off += n_div
        us    = p[off:off+nstate];  off += nstate
        uc    = p[off:off+ncnty]
        return g0, sig, gc, gp, gd, gs, ug, ur, ue, ureg, us, uc

    def neg_lp(p):
        g0, sig, gc, gp, gd, gs, ug, ur, ue, ureg, us, uc = _unpack(p)
        sgr, sr, se, sc, ss, sreg = sig
        eta  = g0 + ug[gi] + ur[ri] + ue[ei] + uc[ci]
        ll   = np.sum(y_tr * eta - np.logaddexp(0.0, eta))
        mu_c = us[county_state_idx] + gc*co2_std + gp*pres_std + gd*drive_std + gs*samesex_std
        mu_s = ureg[state_div_idx]
        lp_c   = -0.5 * np.sum((uc - mu_c)**2) / sc**2 - ncnty * np.log(sc)
        lp_s   = -0.5 * np.sum((us - mu_s)**2) / ss**2 - nstate * np.log(ss)
        lp_reg = -0.5 * np.sum(ureg**2) / sreg**2 - n_div * np.log(sreg)
        lp_g   = -0.5 * np.sum(ug**2) / sgr**2 - ng * np.log(sgr)
        lp_r   = -0.5 * np.sum(ur**2) / sr**2  - nr * np.log(sr)
        lp_e   = -0.5 * np.sum(ue**2) / se**2  - ne * np.log(se)
        lp_hyp = (-0.5 * g0**2 / 1.5**2
                  - 0.5 * (gc**2 + gp**2 + gd**2 + gs**2)
                  + np.sum(-0.5 * sig**2 / 2.5**2 + p[1:7]))
        return -(ll + lp_c + lp_s + lp_reg + lp_g + lp_r + lp_e + lp_hyp)

    n_p = 1 + 6 + 4 + ng + nr + ne + n_div + nstate + ncnty
    x0  = np.zeros(n_p)
    x0[1:7] = np.log(0.5)
    res = minimize(neg_lp, x0, method="L-BFGS-B",
                   options={"maxiter": 3000, "ftol": 1e-9, "gtol": 1e-6})
    g0, _, gc, gp, gd, gs, ug, ur, ue, ureg, us, uc = _unpack(res.x)

    def predict_hlm(X):
        gi_ = X[:, 0].astype(int);  ri_ = X[:, 1].astype(int)
        ei_ = X[:, 2].astype(int);  ci_ = X[:, 3].astype(int)
        return expit(g0 + ug[gi_] + ur[ri_] + ue[ei_] + uc[ci_])

    return predict_hlm


def fit_lasso(y_tr, X_tr_ohe, X_ps_ohe_ref):
    m = LassoCV(cv=5, random_state=SEED, max_iter=5000).fit(X_tr_ohe, y_tr)
    return m.predict


def fit_knn(y_tr, X_tr):
    # Label-encoded (county_fips integer-coded) — one-hot of 3,143 counties
    # would slow KNN to a crawl, so we use label encoding for the county only.
    best_k, best_mse = 1, np.inf
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    for k in range(1, 202):
        preds = np.zeros(len(y_tr))
        for tr, va in kf.split(X_tr):
            m = KNeighborsRegressor(n_neighbors=k)
            m.fit(X_tr[tr], y_tr[tr])
            preds[va] = m.predict(X_tr[va])
        mse = np.mean((y_tr - preds) ** 2)
        if mse < best_mse:
            best_mse, best_k = mse, k
    m = KNeighborsRegressor(n_neighbors=best_k).fit(X_tr, y_tr)
    print(f"  KNN best k={best_k}")
    return m.predict


def fit_rf(y_tr, X_tr):
    m = RandomForestRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
    m.fit(X_tr, y_tr)
    return m.predict


def fit_xgb(y_tr, X_tr):
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    cv_res = xgb.cv(
        params={"objective": "reg:squarederror", "eval_metric": "rmse",
                "eta": 0.02, "seed": SEED},
        dtrain=dtrain, num_boost_round=int(50 / 0.02),
        nfold=5, early_stopping_rounds=20, verbose_eval=False,
    )
    best_n = int(cv_res["test-rmse-mean"].idxmin()) + 1
    m = xgb.train(
        params={"objective": "reg:squarederror", "eval_metric": "rmse",
                "eta": 0.02, "seed": SEED},
        dtrain=dtrain, num_boost_round=best_n, verbose_eval=False,
    )
    print(f"  XGBoost best n_rounds={best_n}")
    return lambda X: m.predict(xgb.DMatrix(X))

print("Base learner helpers defined.")

Base learner helpers defined.


### Learn stacking weights via 5-fold CV

In [9]:
start = time.time()
kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros((len(y), 5))

for fold, (tr, va) in enumerate(kf.split(X_label)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    y_tr, y_va = y[tr], y[va]
    Xl_tr, Xl_va = X_label[tr], X_label[va]
    Xo_tr, Xo_va = X_ohe[tr],   X_ohe[va]

    gi_tr = Xl_tr[:, 0].astype(int); ri_tr = Xl_tr[:, 1].astype(int)
    ei_tr = Xl_tr[:, 2].astype(int); ci_tr = Xl_tr[:, 3].astype(int)

    pred_hlm   = fit_hlm(y_tr, gi_tr, ri_tr, ei_tr, ci_tr, N_GROUPS)
    pred_lasso = fit_lasso(y_tr, Xo_tr, Xo_va)
    pred_knn   = fit_knn(y_tr, Xl_tr)
    pred_rf    = fit_rf(y_tr, Xl_tr)
    pred_xgb   = fit_xgb(y_tr, Xl_tr)

    oof[va, 0] = pred_hlm(Xl_va)
    oof[va, 1] = pred_lasso(Xo_va)
    oof[va, 2] = pred_knn(Xl_va)
    oof[va, 3] = pred_rf(Xl_va)
    oof[va, 4] = pred_xgb(Xl_va)

stack_weights, _ = nnls(oof, y)
if stack_weights.sum() > 0:
    stack_weights = stack_weights / stack_weights.sum()

print("\nStack weights (HLM / LASSO / KNN / RF / XGB):")
for name, w in zip(["HLM", "LASSO", "KNN", "RF", "XGBoost"], stack_weights):
    print(f"  {name:<8} {w:.4f}")
print(f"\nCV stacking complete in {(time.time()-start)/60:.1f} min")

Fold 1/5
  KNN best k=195
  XGBoost best n_rounds=19
Fold 2/5
  KNN best k=91
  XGBoost best n_rounds=1
Fold 3/5
  KNN best k=192
  XGBoost best n_rounds=1
Fold 4/5
  KNN best k=121
  XGBoost best n_rounds=5
Fold 5/5
  KNN best k=40
  XGBoost best n_rounds=2

Stack weights (HLM / LASSO / KNN / RF / XGB):
  HLM      0.0000
  LASSO    0.7828
  KNN      0.1508
  RF       0.0665
  XGBoost  0.0000

CV stacking complete in 0.2 min


### Retrain on full data → predict on county poststrat frame

In [10]:
print("Retraining on full dataset...")
gi = X_label[:, 0].astype(int); ri = X_label[:, 1].astype(int)
ei = X_label[:, 2].astype(int); ci = X_label[:, 3].astype(int)

pred_hlm_full   = fit_hlm(y, gi, ri, ei, ci, N_GROUPS)
pred_lasso_full = fit_lasso(y, X_ohe, ps_ohe)
pred_knn_full   = fit_knn(y, X_label)
pred_rf_full    = fit_rf(y, X_label)
pred_xgb_full   = fit_xgb(y, X_label)

M = np.column_stack([
    pred_hlm_full(ps_label),
    pred_lasso_full(ps_ohe),
    pred_knn_full(ps_label),
    pred_rf_full(ps_label),
    pred_xgb_full(ps_label),
])
ps["predicted_prob"] = np.clip(M @ stack_weights, 0, 1)

print(f"Predicted probs: min={ps['predicted_prob'].min():.3f}  "
      f"mean={ps['predicted_prob'].mean():.3f}  max={ps['predicted_prob'].max():.3f}")

Retraining on full dataset...
  KNN best k=52
  XGBoost best n_rounds=2
Predicted probs: min=0.000  mean=0.563  max=0.711


In [11]:
# ── Poststratify by county ────────────────────────────────────────────────
estimates = (
    ps.groupby(["county_fips", "state_fips"])
    .apply(lambda g: np.average(g["predicted_prob"], weights=g["N_rounded"]),
           include_groups=False)
    .reset_index(name="happening_estimate")
)
estimates["state_name"] = estimates["state_fips"].map(STATE_NAMES)
estimates = estimates[["county_fips", "state_fips", "state_name", "happening_estimate"]]

print(f"County estimates: {len(estimates):,}")
print(f"Range: [{estimates['happening_estimate'].min():.4f}, "
      f"{estimates['happening_estimate'].max():.4f}]  "
      f"Mean: {estimates['happening_estimate'].mean():.4f}")

County estimates: 3,143
Range: [0.0000, 0.7074]  Mean: 0.5628


In [12]:
est_dir  = OUTPUT_DIR / "estimates"
diag_dir = OUTPUT_DIR / "diagnostics"
est_dir.mkdir(parents=True, exist_ok=True)
diag_dir.mkdir(parents=True, exist_ok=True)

out_path = est_dir / f"{MODEL_NAME}_county_estimates.csv"
estimates.to_csv(out_path, index=False)
print(f"Saved → {out_path}  (rows: {len(estimates):,})")

diag_path = diag_dir / f"{MODEL_NAME}_county_summary.txt"
with open(diag_path, "w") as f:
    f.write("SRP County — Diagnostic Summary\n" + "=" * 55 + "\n\n")
    f.write(f"Outcome: {OUTCOME}\n")
    f.write(f"Counties: {n_county:,}\n")
    f.write(f"Stack weights:\n")
    for name, w in zip(["HLM", "LASSO", "KNN", "RF", "XGBoost"], stack_weights):
        f.write(f"  {name:<8} {w:.4f}\n")
    f.write(f"\nCounty estimates — mean:{estimates['happening_estimate'].mean():.4f}  "
            f"min:{estimates['happening_estimate'].min():.4f}  "
            f"max:{estimates['happening_estimate'].max():.4f}\n")
print(f"Diagnostics → {diag_path}")

Saved → ../outputs/estimates/srp_county_estimates.csv  (rows: 3,143)
Diagnostics → ../outputs/diagnostics/srp_county_summary.txt


In [13]:
# ── State-level rollup diagnostic ──────────────────────────────────────────
# Roll county estimates up to state via population-weighted average, then compare
# with the same model's state-level CSV (from the parallel state notebook).
state_rollup = (
    estimates.merge(
        ps_county.groupby("county_fips")["N_rounded"].sum().reset_index(name="county_pop"),
        on="county_fips", how="left",
    )
    .dropna(subset=["happening_estimate"])
    .groupby("state_fips")
    .apply(lambda g: np.average(g["happening_estimate"], weights=g["county_pop"]),
           include_groups=False)
    .reset_index(name="county_rollup")
)
state_rollup["state_name"] = state_rollup["state_fips"].map(STATE_NAMES)

state_csv = OUTPUT_DIR / "estimates" / f"{STATE_CSV_NAME}"
if state_csv.exists():
    state_est = pd.read_csv(state_csv, dtype={"state_fips": str})
    cmp = state_rollup.merge(state_est[["state_fips", "estimate"]], on="state_fips", how="left")
    cmp["abs_diff"] = (cmp["county_rollup"] - cmp["estimate"]).abs()
    print(f"\nState rollup vs {STATE_CSV_NAME}:")
    print(f"  mean |diff|: {cmp['abs_diff'].mean():.4f}")
    print(f"  max  |diff|: {cmp['abs_diff'].max():.4f}")
    print(cmp[["state_fips", "state_name", "county_rollup", "estimate", "abs_diff"]]
          .sort_values("abs_diff", ascending=False).head(10).to_string(index=False))
else:
    print(f"\nNo state CSV found at {state_csv} — skipping rollup comparison.")


State rollup vs srp_state_estimates.csv:
  mean |diff|: 0.0362
  max  |diff|: 0.1015
state_fips    state_name  county_rollup  estimate  abs_diff
        02        Alaska       0.499498  0.600993  0.101495
        54 West Virginia       0.649898  0.552792  0.097106
        55     Wisconsin       0.671100  0.575902  0.095198
        09   Connecticut       0.508629  0.594822  0.086194
        49          Utah       0.628767  0.559029  0.069738
        01       Alabama       0.503212  0.568465  0.065254
        19          Iowa       0.517689  0.578223  0.060534
        50       Vermont       0.653254  0.595243  0.058011
        53    Washington       0.671180  0.614030  0.057150
        18       Indiana       0.504298  0.557256  0.052959
